# Ex2 · 자동미분과 optimizer

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import torch

from torch import nn

In [ ]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    leaf = x.detach().clone().requires_grad_(True)
    return torch.autograd.grad((leaf**2).sum(), leaf)[0]

In [ ]:
def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    leaf = x.detach().clone().requires_grad_(True)
    (leaf**2).sum().backward()
    return leaf.grad.detach().clone()

In [ ]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    a_leaf = a.detach().clone().requires_grad_(True)
    b_leaf = b.detach().clone().requires_grad_(True)
    return torch.autograd.grad((a_leaf**2 + a_leaf * b_leaf).sum(), (a_leaf, b_leaf))

In [ ]:
from torch.utils.data import DataLoader, Dataset

In [ ]:
class TensorPairDataset(Dataset):
    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        if len(x) != len(y):
            raise ValueError("x and y must contain the same number of samples")
        self.x, self.y = (x, y)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return (self.x[idx], self.y[idx])

In [ ]:
class NextTokenDataset(Dataset):
    def __init__(self, tokens: torch.Tensor):
        if tokens.ndim != 2 or tokens.shape[1] < 2:
            raise ValueError("tokens must have shape (N, T) with T >= 2")
        self.tokens = tokens

    def __len__(self) -> int:
        return len(self.tokens)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return (self.tokens[idx, :-1], self.tokens[idx, 1:])

In [ ]:
class RandomCropSequenceDataset(Dataset):
    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        if tokens.ndim != 2 or not 0 < crop_len <= tokens.shape[1]:
            raise ValueError("crop_len must be in [1, T_total]")
        self.tokens = tokens
        self.crop_len = crop_len
        self.generator = torch.Generator()
        if seed is not None:
            self.generator.manual_seed(seed)

    def __len__(self) -> int:
        return len(self.tokens)

    def __getitem__(self, idx: int) -> torch.Tensor:
        max_start = self.tokens.shape[1] - self.crop_len
        start = int(torch.randint(max_start + 1, (), generator=self.generator))
        return self.tokens[idx, start : start + self.crop_len]

In [ ]:
@dataclass(frozen=True)
class PaddedBatch:
    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    if not seqs:
        raise ValueError("seqs must not be empty")
    device = seqs[0].device
    lengths = torch.tensor([len(seq) for seq in seqs], dtype=torch.long, device=device)
    max_len = int(lengths.max())
    tokens = torch.full(
        (len(seqs), max_len), pad_value, dtype=torch.long, device=device
    )
    for row, seq in enumerate(seqs):
        tokens[row, : len(seq)] = seq.to(device=device, dtype=torch.long)
    positions = torch.arange(max_len, device=device).unsqueeze(0)
    padding_mask = positions >= lengths.unsqueeze(1)
    return PaddedBatch(tokens, lengths, padding_mask)

In [ ]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    inputs, targets = zip(*batch)
    input_batch = pad_1d_sequences(list(inputs), pad_value)
    target_batch = pad_1d_sequences(list(targets), pad_value)
    if not torch.equal(input_batch.lengths, target_batch.lengths):
        raise ValueError("input and target lengths must match")
    return {
        "input_ids": input_batch.tokens,
        "target_ids": target_batch.tokens,
        "attention_mask": ~input_batch.padding_mask,
        "padding_mask": input_batch.padding_mask,
    }

In [ ]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_fn,
        num_workers=num_workers,
    )

In [ ]:
@dataclass
class AdamWState:
    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    with torch.no_grad():
        return AdamWState(torch.zeros_like(p), torch.zeros_like(p), 0)

In [ ]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-08,
    weight_decay: float = 0.01,
) -> AdamWState:
    beta1, beta2 = betas
    with torch.no_grad():
        state.t += 1
        state.m.mul_(beta1).add_(grad, alpha=1 - beta1)
        state.v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        m_hat = state.m / (1 - beta1**state.t)
        v_hat = state.v / (1 - beta2**state.t)
        p.add_(m_hat / (v_hat.sqrt() + eps) + weight_decay * p, alpha=-lr)
    return state

In [ ]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-08,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    if not len(params) == len(grads) == len(states):
        raise ValueError("params, grads, and states must have equal lengths")
    return [
        adamw_step_(p, g, s, lr, betas, eps, weight_decay)
        for p, g, s in zip(params, grads, states, strict=True)
    ]

In [ ]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    model.train()
    x, target = batch
    optimizer.zero_grad()
    loss = torch.nn.functional.mse_loss(model(x), target)
    loss.backward()
    optimizer.step()
    return float(loss.detach())

In [ ]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    if weight.ndim < 2:
        raise ValueError("weight must have at least two dimensions")
    receptive = int(torch.tensor(weight.shape[2:]).prod()) if weight.ndim > 2 else 1
    return (weight.shape[1] * receptive, weight.shape[0] * receptive)

In [ ]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = gain * (6.0 / (fan_in + fan_out)) ** 0.5
    with torch.no_grad():
        return weight.uniform_(-bound, bound)

In [ ]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    if nonlinearity != "relu":
        raise ValueError("this exercise implements the ReLU gain only")
    fan_in, _ = fan_in_fan_out(weight)
    bound = (6.0 / fan_in) ** 0.5
    with torch.no_grad():
        return weight.uniform_(-bound, bound)

In [ ]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    if scheme == "xavier":
        xavier_uniform_(layer.weight)
    elif scheme == "kaiming_relu":
        kaiming_uniform_(layer.weight)
    elif scheme == "zero":
        with torch.no_grad():
            layer.weight.zero_()
    else:
        raise ValueError(f"unknown initialization scheme: {scheme}")
    if layer.bias is not None:
        with torch.no_grad():
            layer.bias.zero_()
    return layer

In [ ]:
padded = pad_1d_sequences([torch.tensor([1, 2]), torch.tensor([3])], pad_value=0)
print(padded.tokens)
print(padded.padding_mask)